In [ ]:
from modules.dataset.make_dataset import make_dataset, dataset_split, NpzDataPipeline
from modules.nn import BranchedCoughCNN, DefaultCNN
import polars as pl
import torch
import torch.optim as optim
from modules.preprocessing.dataframe_process import filter_coughvid_df

In [ ]:
def make_path(audio_name: str) -> str:
    return f"data/public_dataset_v3_wav/{audio_name.split(".")[0]}.wav"

df: pl.DataFrame = filter_coughvid_df(
    pl.read_csv(
        'data/tabular_form/tabular_form/coughvid_v3.csv', 
        schema_overrides={"age":pl.Float32}
    )
)

In [ ]:
# generates dataset

make_dataset(df, 16_000, out="data\\final_processed_dataset.npz")


In [ ]:
dataset: NpzDataPipeline = NpzDataPipeline("data\\final_processed_dataset.npz")
train_dataset, test_dataset, val_dataset = dataset_split(dataset)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, 64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_dataset, 64, shuffle=True, num_workers=4, pin_memory=True)

In [ ]:
model: torch.nn.Module | None = None

In [ ]:
device: torch.types.Device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


In [ ]:
loader

In [ ]:
for epoch in range(num_epochs):
    # Training Phase
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        # Move inputs to GPU. non_blocking=True works with pin_memory=True in DataLoader
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) # Memory Speedup
        
        # Runs the forward pass in mixed precision (float16/float32 mixed)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Scale the loss and do the backward pass
        scaler.scale(loss).backward()
        
        # Step the optimizer through the scaler (unscales gradients behind the scenes)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
    # Step the learning rate scheduler at the end of the epoch
    scheduler.step()
    
    # 4. Optimized Validation Phase (Inference)
    optimized_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    # inference_mode() is faster and uses less memory than no_grad()
    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    # Log metrics
    train_epoch_loss = running_loss / len(train_loader)
    val_epoch_loss = val_loss / len(val_loader)
    val_accuracy = 100.0 * correct / total
    
    print(f"Epoch [{epoch+1}/{num_epochs}] -> "
          f"Train Loss: {train_epoch_loss:.4f} | "
          f"Val Loss: {val_epoch_loss:.4f} | "
          f"Val Acc: {val_accuracy:.2f}%")